# PlantVillage disease classifier - resumable Colab training

This notebook trains a crop/disease classifier on every class in `geraldmc/plantvillage-full`. It supports either MobileNetV3-Large or ResNet-50.

Free-Colab design:
- checkpoints and exports are stored in Google Drive;
- checkpoints are saved after every epoch and periodically during a long epoch;
- rerunning setup and training resumes from `latest.pt`;
- MobileNetV3 is the low-VRAM default;
- ResNet-50 is available by changing `BACKBONE`;
- AMP, 224 px inputs, and the provided batch sizes reduce memory use.

The class names and IDs are derived dynamically from PlantVillage metadata, sorted alphabetically, and saved in `model_config.json` and every checkpoint.

In [ ]:
%pip install -q "pandas==2.2.3" "pyarrow<24,>=19"
%pip install -q datasets huggingface_hub tqdm scikit-learn matplotlib seaborn opencv-python-headless onnx
# Do not upgrade torch or torchvision: keep Colab's CUDA-compatible versions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, random, time, zipfile

# Backbone choice: 'mobilenet_v3' or 'resnet50'.
BACKBONE = 'mobilenet_v3'
if BACKBONE not in {'mobilenet_v3', 'resnet50'}:
    raise ValueError("BACKBONE must be 'mobilenet_v3' or 'resnet50'")

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32 if BACKBONE == 'mobilenet_v3' else 16
NUM_WORKERS = 2
MAX_EPOCHS = 15
HEAD_EPOCHS = 2
MAX_MINUTES_PER_SESSION = 45
CHECKPOINT_EVERY_STEPS = 200

WORK_ROOT = Path('/content/drive/MyDrive/plant-disease/plantvillage_classifier')
RUN_ROOT = WORK_ROOT / f'runs/{BACKBONE}'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('Run directory:', RUN_ROOT)


## 1. Load and map PlantVillage

The dataset is loaded directly from Hugging Face. Every unique `class_label` is included; no crop or disease is filtered out. Split names come from the dataset `split` column, not random resampling.

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from PIL import Image

DATASET_ID = 'geraldmc/plantvillage-full'
dataset = load_dataset(DATASET_ID)

source = dataset[next(iter(dataset.keys()))]
CLASS_NAMES = sorted(set(source['class_label']))
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}
print(f'Training all {len(CLASS_NAMES)} PlantVillage classes')
print('\n'.join(f'{index}: {name}' for index, name in enumerate(CLASS_NAMES)))
print(source.features)
print(pd.Series(source['split']).value_counts().to_dict())

In [ ]:
import collections
from sklearn.model_selection import train_test_split

def normalize_split(value):
    value = str(value).lower()
    if value in {'train', 'training'}:
        return 'train'
    if value in {'validation', 'val', 'valid'}:
        return 'val'
    if value in {'test', 'testing'}:
        return 'test'
    raise ValueError(f'Unknown split value: {value}')

split_rows = {name: [] for name in ('train', 'val', 'test')}
for index, row in enumerate(source):
    label = row['class_label']
    split = normalize_split(row['split'])
    split_rows[split].append((index, CLASS_TO_ID[label]))

# PlantVillage metadata may provide train/test only. Create a fixed stratified holdout.
if not split_rows['val']:
    train_labels = [class_id for _, class_id in split_rows['train']]
    split_rows['train'], split_rows['val'] = train_test_split(
        split_rows['train'],
        test_size=0.15,
        random_state=SEED,
        shuffle=True,
        stratify=train_labels,
    )

for split, rows in split_rows.items():
    counts = collections.Counter(class_id for _, class_id in rows)
    print(split, len(rows), 'per-class min/max:',
          min(counts.values()) if counts else 0,
          max(counts.values()) if counts else 0)

if any(len(split_rows[name]) == 0 for name in ('train', 'val', 'test')):
    raise RuntimeError('Train, validation, and test partitions must all be non-empty')


## 2. Dataset, augmentation, and class weighting

Augmentation is training-only. Validation and test images are resized without random transforms. If one class is substantially larger, inverse-frequency class weights and a weighted sampler help reduce majority-class bias.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF

class PlantVillageDataset(Dataset):
    def __init__(self, hf_dataset, rows, training=False):
        self.hf_dataset = hf_dataset
        self.rows = rows
        self.training = training

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, position):
        source_index, class_id = self.rows[position]
        image = self.hf_dataset[source_index]['image'].convert('RGB')
        image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        if self.training:
            if random.random() < 0.5:
                image = TF.hflip(image)
            if random.random() < 0.3:
                image = TF.vflip(image)
            if random.random() < 0.35:
                image = TF.adjust_brightness(image, random.uniform(0.75, 1.25))
            if random.random() < 0.35:
                image = TF.adjust_contrast(image, random.uniform(0.75, 1.25))
            if random.random() < 0.25:
                image = TF.adjust_saturation(image, random.uniform(0.8, 1.2))
        tensor = TF.to_tensor(image)
        tensor = TF.normalize(tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return tensor, torch.tensor(class_id, dtype=torch.long)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_counts = np.bincount([class_id for _, class_id in split_rows['train']], minlength=len(CLASS_NAMES)).astype(np.float64)
class_weights = train_counts.sum() / np.maximum(train_counts * len(CLASS_NAMES), 1.0)
sample_weights = [class_weights[class_id] for _, class_id in split_rows['train']]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
generator = torch.Generator().manual_seed(SEED)

train_dataset = PlantVillageDataset(source, split_rows['train'], training=True)
val_dataset = PlantVillageDataset(source, split_rows['val'])
test_dataset = PlantVillageDataset(source, split_rows['test'])

loaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                       num_workers=NUM_WORKERS, pin_memory=True,
                       persistent_workers=NUM_WORKERS > 0, worker_init_fn=seed_worker,
                       generator=generator, drop_last=True),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True,
                     persistent_workers=NUM_WORKERS > 0),
    'test': DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=NUM_WORKERS > 0),
}
print({name: len(loader.dataset) for name, loader in loaders.items()})

## 3. Model and resume setup

Training is two-stage: first train a new classifier head with the pretrained backbone frozen, then fully fine-tune all parameters with a lower learning rate. Checkpoints include model, optimizer, scheduler, AMP scaler, epoch, history, and best validation result.

In [ ]:
import torch.nn as nn
import torchvision
from torchvision.models import (
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
    resnet50, ResNet50_Weights,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if BACKBONE == 'mobilenet_v3':
    model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(CLASS_NAMES))
else:
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model.to(device)

criterion = nn.CrossEntropyLoss(weight=torch.from_numpy(class_weights).float().to(device))

def set_backbone_training(model, enabled):
    if BACKBONE == 'mobilenet_v3':
        for parameter in model.features.parameters():
            parameter.requires_grad = enabled
    else:
        for name, parameter in model.named_parameters():
            if not name.startswith('fc.'):
                parameter.requires_grad = enabled

set_backbone_training(model, False)

def build_optimizer(fine_tuning):
    if fine_tuning:
        set_backbone_training(model, True)
        if BACKBONE == 'mobilenet_v3':
            backbone_parameters = model.features.parameters()
            head_parameters = model.classifier.parameters()
        else:
            backbone_parameters = (
                parameter for name, parameter in model.named_parameters()
                if not name.startswith('fc.') and parameter.requires_grad
            )
            head_parameters = model.fc.parameters()
        return torch.optim.AdamW([
            {'params': backbone_parameters, 'lr': 1e-4},
            {'params': head_parameters, 'lr': 1e-3},
        ], weight_decay=1e-4)

    set_backbone_training(model, False)
    return torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=1e-3,
        weight_decay=1e-4,
    )

optimizer = build_optimizer(False)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

def save_checkpoint(next_epoch, best_val_acc, history, global_step, name='latest.pt'):
    payload = {
        'next_epoch': next_epoch,
        'best_val_acc': best_val_acc,
        'history': history,
        'global_step': global_step,
        'backbone': BACKBONE,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'class_names': CLASS_NAMES,
        'config': {
            'image_size': IMAGE_SIZE,
            'normalization_mean': [0.485, 0.456, 0.406],
            'normalization_std': [0.229, 0.224, 0.225],
            'dataset': DATASET_ID,
            'num_classes': len(CLASS_NAMES),
        },
    }
    temporary = RUN_ROOT / f'.{name}.tmp'
    torch.save(payload, temporary)
    temporary.replace(RUN_ROOT / name)

history = []
best_val_acc = -1.0
global_step = 0
start_epoch = 0
latest = RUN_ROOT / 'latest.pt'
if latest.exists():
    state = torch.load(latest, map_location=device)
    if state.get('backbone') != BACKBONE:
        raise RuntimeError(f"Checkpoint backbone {state.get('backbone')} != selected {BACKBONE}")
    if state.get('class_names') != CLASS_NAMES:
        raise RuntimeError('Checkpoint class taxonomy differs from this dataset run')

    model.load_state_dict(state['model'])
    history = state.get('history', [])
    best_val_acc = state.get('best_val_acc', -1.0)
    global_step = state.get('global_step', 0)
    start_epoch = state.get('next_epoch', 0)

    fine_tuning = start_epoch >= HEAD_EPOCHS
    optimizer = build_optimizer(fine_tuning)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(MAX_EPOCHS - start_epoch, 1),
    )
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    scaler.load_state_dict(state['scaler'])
    print(f'Resuming epoch {start_epoch + 1}; best val accuracy={best_val_acc:.4f}')
else:
    print(f'Starting pretrained {BACKBONE}; backbone frozen for the first {HEAD_EPOCHS} epochs')

print(device, f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters')


## 4. Train

The cell automatically completes or resumes epochs. At `HEAD_EPOCHS`, it unfreezes the backbone and recreates the optimizer with a 10x lower backbone learning rate where applicable.

In [ ]:
from tqdm.auto import tqdm

session_deadline = time.time() + MAX_MINUTES_PER_SESSION * 60
current_epoch = start_epoch
stage_initialized = start_epoch >= HEAD_EPOCHS

def configure_optimizer(epoch):
    if epoch == HEAD_EPOCHS:
        set_backbone_training(model, True)
        if BACKBONE == 'mobilenet_v3':
            backbone_parameters = model.features.parameters()
            head_parameters = model.classifier.parameters()
        else:
            backbone_parameters = (p for n, p in model.named_parameters() if not n.startswith('fc.') and p.requires_grad)
            head_parameters = model.fc.parameters()
        groups = [
            {'params': backbone_parameters, 'lr': 1e-4},
            {'params': head_parameters, 'lr': 1e-3},
        ]
        return torch.optim.AdamW(groups, weight_decay=1e-4)
    return None

try:
    for epoch in range(start_epoch, MAX_EPOCHS):
        current_epoch = epoch
        if epoch == HEAD_EPOCHS and not stage_initialized:
            optimizer = build_optimizer(fine_tuning=True)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(MAX_EPOCHS - epoch, 1))
            stage_initialized = True

        model.train()
        running_loss = 0.0
        progress = tqdm(loaders['train'], desc=f'Epoch {epoch + 1}/{MAX_EPOCHS}')
        for images, labels in progress:
            if time.time() >= session_deadline:
                save_checkpoint(epoch, best_val_acc, history, global_step)
                print('Session time limit reached; resume later from latest.pt.')
                raise TimeoutError

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            global_step += 1
            progress.set_postfix(loss=f'{loss.item():.4f}')
            if global_step % CHECKPOINT_EVERY_STEPS == 0:
                save_checkpoint(epoch, best_val_acc, history, global_step)

        model.eval()
        correct = 0
        total = 0
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in loaders['val']:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    logits = model(images)
                    val_loss += criterion(logits, labels).item()
                correct += (logits.argmax(1) == labels).sum().item()
                total += labels.numel()

        val_acc = correct / max(total, 1)
        row = {
            'epoch': epoch + 1,
            'train_loss': running_loss / max(len(loaders['train']), 1),
            'val_loss': val_loss / max(len(loaders['val']), 1),
            'val_accuracy': val_acc,
        }
        history.append(row)
        scheduler.step()

        save_checkpoint(epoch + 1, max(best_val_acc, val_acc), history, global_step)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(epoch + 1, best_val_acc, history, global_step, 'best.pt')
        print(row)
except TimeoutError:
    pass
except KeyboardInterrupt:
    save_checkpoint(current_epoch, best_val_acc, history, global_step)
    raise

print(f'Completed through epoch {history[-1]["epoch"] if history else start_epoch}.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if history:
    history_df = pd.DataFrame(history)
    display(history_df)
    history_df.to_csv(RUN_ROOT / 'history.csv', index=False)
    history_df.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o')
    plt.grid(); plt.show()
    history_df.plot(x='epoch', y='val_accuracy', marker='o', ylim=(0, 1))
    plt.grid(); plt.show()
else:
    print('No completed epoch yet. Rerun the training cell.')

## 5. Test, metrics, and export

The test set is evaluated only after training. The exports are a PyTorch checkpoint, TorchScript model, ONNX model, class configuration, and a zip package.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
import seaborn as sns

best = RUN_ROOT / 'best.pt'
if not best.exists():
    raise FileNotFoundError('No best.pt yet; complete at least one validation epoch')
state = torch.load(best, map_location=device)
model.load_state_dict(state['model'])
model.eval()

all_labels = []
all_predictions = []
all_probabilities = []
with torch.no_grad():
    for images, labels in tqdm(loaders['test'], desc='test'):
        images = images.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
            logits = model(images)
        probabilities = logits.float().softmax(1).cpu().numpy()
        all_probabilities.append(probabilities)
        all_predictions.append(probabilities.argmax(1))
        all_labels.append(labels.numpy())

y_true = np.concatenate(all_labels)
y_pred = np.concatenate(all_predictions)
y_score = np.concatenate(all_probabilities)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))
print('Top-1 accuracy:', (y_true == y_pred).mean())
print('Top-3 accuracy:', top_k_accuracy_score(y_true, y_score, k=3, labels=np.arange(len(CLASS_NAMES))))

matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(matrix, cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
plt.savefig(RUN_ROOT / 'confusion_matrix.png', dpi=150)
plt.show()

(RUN_ROOT / 'test_metrics.json').write_text(json.dumps({
    'top1_accuracy': float((y_true == y_pred).mean()),
    'top3_accuracy': float(top_k_accuracy_score(y_true, y_score, k=3, labels=np.arange(len(CLASS_NAMES)))),
    'backbone': BACKBONE,
    'dataset': DATASET_ID,
    'class_names': CLASS_NAMES,
}, indent=2) + '\n')

In [ ]:
model.eval().cpu()
example = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
with torch.no_grad():
    traced = torch.jit.trace(model, example)
traced.save(str(RUN_ROOT / 'plantvillage_classifier.ts'))

(RUN_ROOT / 'model_config.json').write_text(json.dumps({
    'task': 'crop_disease_classification',
    'backbone': BACKBONE,
    'input_size': [IMAGE_SIZE, IMAGE_SIZE],
    'normalization_mean': [0.485, 0.456, 0.406],
    'normalization_std': [0.229, 0.224, 0.225],
    'class_names': CLASS_NAMES,
    'dataset': DATASET_ID,
    'license': 'Verify PlantVillage/Hugging Face dataset terms before redistribution',
}, indent=2) + '\n')

torch.save(model.state_dict(), RUN_ROOT / 'plantvillage_classifier_state_dict.pt')

try:
    torch.onnx.export(
        model,
        example,
        str(RUN_ROOT / 'plantvillage_classifier.onnx'),
        input_names=['image'],
        output_names=['logits'],
        dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=17,
        do_constant_folding=True,
    )
except Exception as error:
    print('ONNX export skipped:', error)

package = WORK_ROOT / f'plantvillage_{BACKBONE}_package.zip'
with zipfile.ZipFile(package, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in (
        'plantvillage_classifier.ts', 'plantvillage_classifier_state_dict.pt',
        'plantvillage_classifier.onnx', 'model_config.json', 'best.pt',
        'latest.pt', 'history.csv', 'test_metrics.json', 'confusion_matrix.png',
    ):
        path = RUN_ROOT / name
        if path.exists():
            archive.write(path, name)
print('Export package:', package)


## 6. Inference contract

For app inference:

1. Load `plantvillage_classifier.ts` or `plantvillage_classifier.onnx`.
2. Resize to 224 x 224 and normalize with `model_config.json`.
3. Apply `softmax(logits, dim=1)`.
4. Get the highest-probability class and top-k predictions.
5. Add a leaf/non-leaf or low-confidence guard if the app accepts arbitrary photos.
6. Combine disease classification with the separate PlantSeg lesion mask in the app API.

This model has only PlantVillage crop/disease classes and is trained mainly on PlantVillage-style images. It is not a field-ready universal plant diagnosis model.